In [2]:
import sqlite3
conn = sqlite3.connect("data/databases/employee.db")
cursor = conn.cursor() # cursor will point to my company.db database and will be used for insertion & deletion of records, creating tables, etc.

# Creating a table in employee.db with the help of cursor:


cursor.execute('''CREATE TABLE IF NOT EXISTS EMPLOYEE (Id INTEGER PRIMARY KEY , Name TEXT, Role TEXT, Department TEXT, Salary REAL)''')

cursor.execute('''CREATE TABLE IF NOT EXISTS PROJECTS (Id INTEGER PRIMARY KEY, Name TEXT, Status TEXT, Budget REAL, Lead_Id INTEGER)''')

##Creating data for insertion:

employees = [
    (1,"Jaywardhan Pagar", "Data Scientist", "Data Science", 150000),
    (2,"Anurag Mhaske", "Software Developer", "Development", 120000),
    (3,"Nakul Karule", "Dev-Ops Engineer", "Dev-Ops", 170000),
    (4,"Maulik Tondawal", "Data Scientist", "Data Science", 140000),
    (5,"Antriksh Soun", "Designer", "Designing", 100000)
]

projects = [
    (1,"RAG Based AI Teaching Assistant","Completed","250000",1),
    (2,"RAG Based Document Management System","Active","400000",4),
    (3,"Neural Network for Breast Cancer", "Completed","600000",1),
    (4,"Fighter Drone","Active","500000",5),
    (5,"Medical AI Assistant","Discontinued","300000",3)
]

cursor.executemany('INSERT OR REPLACE INTO EMPLOYEE VALUES(?,?,?,?,?)',employees)
cursor.executemany('INSERT OR REPLACE INTO PROJECTS VALUES(?,?,?,?,?)',projects)

conn.commit()

conn.close()

In [4]:
## Custom parsing of database:

from typing import List
from langchain_core.documents import Document
import sqlite3

def Customsql(sqlpath: str) -> List[Document]:

    """Convert SQL Database to Document Data Structures with context"""

    conn = sqlite3.connect(sqlpath)
    cursor = conn.cursor()

    documents = []

    ## Step1: Create documents for each table

    cursor.execute("SELECT name from sqlite_master WHERE type = 'table';")
    tables = cursor.fetchall()

    for table in tables:

        table_name = table[0]

        #Get table schema:

        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()

        column_names = [col[1] for col in columns]

        # Get table data:

        cursor.execute(f"select * from {table_name}") 
        rows = cursor.fetchall()

        #Create table overview document:

        table_content = f"Table: {table_name}\n"
        table_content += f"Columns: {",".join(column_names)}\n"
        table_content += f"Total Records: {len(rows)}\n"

        # Add sample records:

        table_content += "Sample Records: \n"

        for row in rows[:5]:
            records = dict(zip(column_names,row))
            table_content += f"\n{records}\n"


        doc = Document(
            page_content= table_content,
            metadata = {
                'source' : sqlpath,
                'table_name' : table_name,
                'num_records' : len(rows),
                'data_type' : 'sql_table'
            }
        )

        documents.append(doc)
    conn.close()
    return documents


docs = Customsql("data/databases/employee.db")
print(docs[0])



page_content='Table: EMPLOYEE
Columns: Id,Name,Role,Department,Salary
Total Records: 5
Sample Records: 

{'Id': 1, 'Name': 'Jaywardhan Pagar', 'Role': 'Data Scientist', 'Department': 'Data Science', 'Salary': 150000.0}

{'Id': 2, 'Name': 'Anurag Mhaske', 'Role': 'Software Developer', 'Department': 'Development', 'Salary': 120000.0}

{'Id': 3, 'Name': 'Nakul Karule', 'Role': 'Dev-Ops Engineer', 'Department': 'Dev-Ops', 'Salary': 170000.0}

{'Id': 4, 'Name': 'Maulik Tondawal', 'Role': 'Data Scientist', 'Department': 'Data Science', 'Salary': 140000.0}

{'Id': 5, 'Name': 'Antriksh Soun', 'Role': 'Designer', 'Department': 'Designing', 'Salary': 100000.0}
' metadata={'source': 'data/databases/employee.db', 'table_name': 'EMPLOYEE', 'num_records': 5, 'data_type': 'sql_table'}
